In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

file_rotterdamPortData = r'..\data\rotterdam_dataset_16.xlsx'
wd = pd.read_excel(file_rotterdamPortData)
wd.info()
wd.head()

In [ ]:
# Distributional shape of cargo per vessel (all vessels together)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

wd['Cargo_tons'].hist(bins=60, ax=ax[0])
ax[0].set_title('Cargo per vessel (all types)')
ax[0].set_xlabel('Cargo (tons)')
ax[0].set_ylabel('Count')

np.log(wd['Cargo_tons']).hist(bins=60, ax=ax[1])
ax[1].set_title('log(Cargo per vessel)')
ax[1].set_xlabel('log(tons)')
ax[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

# the overall histogram is a mixture of three vessel types, so we analyse each type separately

In [ ]:
# Descriptive statistics and skewness per vessel type
types = ['Container', 'Tanker', 'Bulk']

stats_table = wd.groupby('Vessel_Type')['Cargo_tons'].agg(['count', 'mean', 'median', 'std', 'min', 'max', 'skew', 'kurt'])
stats_table['CV'] = stats_table['std'] / stats_table['mean']
print(stats_table.round(2))

# skewness of log(cargo): close to 0 means log(cargo) is symmetric (supports lognormal)
wd['log_cargo'] = np.log(wd['Cargo_tons'])
print(wd.groupby('Vessel_Type')['log_cargo'].agg(['mean', 'std', 'skew']).round(3))

# all three types are right skewed (skew > 0, mean > median)

In [ ]:
# Differences between vessel categories
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

for t in types:
    wd.loc[wd['Vessel_Type'] == t, 'Cargo_tons'].hist(bins=60, density=True, alpha=0.5, ax=ax[0], label=t)
ax[0].set_title('Cargo per vessel by type')
ax[0].set_xlabel('Cargo (tons)')
ax[0].set_ylabel('Density')
ax[0].legend()

wd.boxplot(column='Cargo_tons', by='Vessel_Type', ax=ax[1])
ax[1].set_title('Cargo per vessel by type')
ax[1].set_ylabel('Cargo (tons)')
plt.suptitle('')
plt.tight_layout()
plt.show()

# test whether the cargo distributions differ between types (Kruskal-Wallis, no normality needed)
groups = [wd.loc[wd['Vessel_Type'] == t, 'Cargo_tons'] for t in types]
print(stats.kruskal(*groups))

In [ ]:
# Fit candidate distributions per vessel type
# candidates: lognormal, gamma, Weibull, normal (as a reference)
# parameters estimated by maximum likelihood (stats.fit), location fixed at 0 for the positive distributions

candidates = {'Lognormal': stats.lognorm,
              'Gamma': stats.gamma,
              'Weibull': stats.weibull_min,
              'Normal': stats.norm}

fits = {}
rows = []
for t in types:
    x = wd.loc[wd['Vessel_Type'] == t, 'Cargo_tons'].values
    for name in candidates:
        dist = candidates[name]
        if name == 'Normal':
            params = dist.fit(x)
            k = 2
        else:
            params = dist.fit(x, floc=0)
            k = 2   # loc is fixed, so only 2 free parameters
        fits[(t, name)] = params

        logL = np.sum(dist.logpdf(x, *params))
        aic = 2 * k - 2 * logL
        ks = stats.kstest(x, dist.cdf, args=params)
        rows.append([t, name, logL, aic, ks.statistic, ks.pvalue])

fit_table = pd.DataFrame(rows, columns=['Type', 'Distribution', 'logL', 'AIC', 'KS stat', 'KS p-value'])
print(fit_table.round(4).to_string(index=False))

# lower AIC = better fit; KS p-value > 0.05 means no evidence against the distribution

In [ ]:
# Method of moments for the lognormal (needed for the basic scenario in 4.6)
# if X ~ Lognormal(mu, sigma^2):  E[X] = exp(mu + sigma^2/2),  Var[X] = (exp(sigma^2) - 1) exp(2mu + sigma^2)
# solving gives:  sigma^2 = log(1 + s^2/m^2),  mu = log(m) - sigma^2/2

rows = []
for t in types:
    x = wd.loc[wd['Vessel_Type'] == t, 'Cargo_tons'].values
    m = np.mean(x)
    s2 = np.var(x)
    sigma2 = np.log(1 + s2 / m**2)
    mu = np.log(m) - sigma2 / 2
    sigma = np.sqrt(sigma2)

    # in scipy: lognorm(s=sigma, scale=exp(mu))
    ks = stats.kstest(x, stats.lognorm(s=sigma, scale=np.exp(mu)).cdf)
    rows.append([t, mu, sigma, ks.pvalue])

mom_table = pd.DataFrame(rows, columns=['Type', 'mu (MoM)', 'sigma (MoM)', 'KS p-value'])
print(mom_table.round(4).to_string(index=False))

In [ ]:
# Histogram with fitted densities per vessel type
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

for i, t in enumerate(types):
    x = wd.loc[wd['Vessel_Type'] == t, 'Cargo_tons'].values
    ax[i].hist(x, bins=60, density=True, alpha=0.5, label='Observed')
    xs = np.linspace(0, x.max(), 500)
    for name in candidates:
        ax[i].plot(xs, candidates[name].pdf(xs, *fits[(t, name)]), lw=1.5, label=name)
    ax[i].set_title(t)
    ax[i].set_xlabel('Cargo (tons)')
    ax[i].set_ylabel('Density')
ax[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
# QQ-plots: best way to see how well the tails are fitted
fig, ax = plt.subplots(3, 3, figsize=(13, 12))

for i, t in enumerate(types):
    x = wd.loc[wd['Vessel_Type'] == t, 'Cargo_tons'].values
    for j, name in enumerate(['Lognormal', 'Gamma', 'Weibull']):
        stats.probplot(x, dist=candidates[name](*fits[(t, name)]), plot=ax[i, j])
        ax[i, j].set_title(t + ' - ' + name)
plt.tight_layout()
plt.show()

In [ ]:
# Extreme observations
# outliers based on the IQR rule on log scale (because of the skewness)
for t in types:
    x = wd.loc[wd['Vessel_Type'] == t, 'Cargo_tons']
    lx = np.log(x)
    q1, q3 = np.percentile(lx, [25, 75])
    iqr = q3 - q1
    n_out = np.sum((lx < q1 - 1.5 * iqr) | (lx > q3 + 1.5 * iqr))
    print(t, '- outliers (log-IQR rule):', n_out, 'of', len(x))

# largest vessels per type
print(wd.sort_values('Cargo_tons', ascending=False).groupby('Vessel_Type').head(3)[['Vessel_ID', 'Vessel_Type', 'Cargo_tons', 'Berth_Time_hr']])

In [ ]:
# How important are the extremes?
# 1) share of the total cargo carried by the largest 1% of vessels
# 2) effect on the mean when removing the top 1%
# 3) compare observed tail probabilities with the fitted distributions

rows = []
for t in types:
    x = np.sort(wd.loc[wd['Vessel_Type'] == t, 'Cargo_tons'].values)
    q99 = np.percentile(x, 99)
    top = x[x > q99]
    share_top = np.sum(top) / np.sum(x)
    mean_without = np.mean(x[x <= q99])

    p_emp = np.mean(x > q99)
    p_logn = 1 - stats.lognorm.cdf(q99, *fits[(t, 'Lognormal')])
    p_gamma = 1 - stats.gamma.cdf(q99, *fits[(t, 'Gamma')])
    p_weib = 1 - stats.weibull_min.cdf(q99, *fits[(t, 'Weibull')])
    rows.append([t, q99, share_top, np.mean(x), mean_without, p_emp, p_logn, p_gamma, p_weib])

ext_table = pd.DataFrame(rows, columns=['Type', '99% quantile', 'Share top 1%', 'Mean', 'Mean w/o top 1%',
                                        'P(X>q99) data', 'Lognormal', 'Gamma', 'Weibull'])
print(ext_table.round(4).to_string(index=False))

In [ ]:
# Relation between cargo and berth time (useful for interpreting extremes operationally)
fig, ax = plt.subplots(figsize=(8, 5))
for t in types:
    sub = wd[wd['Vessel_Type'] == t]
    ax.scatter(sub['Cargo_tons'], sub['Berth_Time_hr'], s=3, alpha=0.4, label=t)
ax.set_xlabel('Cargo (tons)')
ax.set_ylabel('Berth time (hours)')
ax.legend()
plt.show()

print(wd.groupby('Vessel_Type')[['Cargo_tons', 'Berth_Time_hr']].corr().round(3))